
### Query Orders Data as JSON Strings

- Extract Top Level Column Values
- Extract Array Elements
- Extract Nested Column Values
- Cast Column Values to specific Data Type

In [0]:
%sql
select * from gizmobox.bronze.v_orders;

#### Extract Top level Object Values


In [0]:
%sql
select value:order_id as order_id,
       value:customer_id as customer_id,
       value:order_date as order_date,
       value:transaction_timestamp as transaction_timestamp,
       value:total_amount as total_amount,
       value:payment_method as payment_method,
       value:items as items,
       value:order_status as order_status,
 value from gizmobox.bronze.v_orders

#### Extract Array Elements

In [0]:
%sql
select value:items[0] as item_1,
       value:items[1] as item_2,
value
 from gizmobox.bronze.v_orders

#### Extract Nested Column Values 

To Extract the nested fields, use dot notation <column_name>:<extraction_path>[index].leafnode

In [0]:
%sql
select value:items[0].item_id::integer as item_1,
       value:items[1] as item_2,
value
 from gizmobox.bronze.v_orders

### Tranform Orders Data - String to JSON

- Pre-process the JSON String to fix the Data Quality Issues

- Transfrom JSON String to JSON Object

- Write Transformed data to the Silver Layer

In [0]:
%sql
select * from gizmobox.bronze.v_orders

Databricks data profile. Run in Databricks to view.

In [0]:
%sql
-- select value,
--         regexp_replace(value,'"order_date":(\\d{4}-\\d{2}-\\d{2})','"order_date":"\$1"') as fixed_value
-- from gizmobox.bronze.v_orders;
select value,
       regexp_replace(value, '"order_date":\\s*(\\d{4}-\\d{2}-\\d{2})', '"order_date":"\$1"') as fixed_value
from gizmobox.bronze.v_orders;

In [0]:
%sql
create or replace temp view tv_orders_fixed 
as
select value,
       regexp_replace(value, '"order_date":\\s*(\\d{4}-\\d{2}-\\d{2})', '"order_date":"\$1"') as fixed_value
from gizmobox.bronze.v_orders;

In [0]:
%sql
select * from tv_orders_fixed

#### Transfrom JSON String to JSON Object
 
 - Function [Schema_of_Json](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/schema_of_json)
 - Function [from_json](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/from_json)


In [0]:
%sql
select schema_of_json(fixed_value) as schema, fixed_value from tv_orders_fixed;

In [0]:
%sql
select from_json(fixed_value,'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>') as json_value
from tv_orders_fixed


In [0]:
%sql
CREATE OR REPLACE TABLE gizmobox.silver.orders_json
AS
select from_json(fixed_value,'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>') as json_value
from tv_orders_fixed

### Transform Orders Data - Explode Arrays

- Access Elements from the JSON Object
- Deduplicate Array Elements
- Explode Arrays
- Write Transformed Data into Silver Layer



In [0]:
%sql
select * from gizmobox.silver.orders_json;

In [0]:
%sql
select 
      json_value.order_id,
      json_value.order_status,
      json_value.payment_method,
      json_value.total_amount,
      json_value.transaction_timestamp,
      json_value.customer_id,
      json_value.items
from gizmobox.silver.orders_json

[Array_distinct](https://docs.databricks.com/aws/en/sql/language-manual/functions/array_distinct)

In [0]:
%sql
CREATE or REPLACE temp view tv_orders_exploded
as 
select 
      json_value.order_id,
      json_value.order_status,
      json_value.payment_method,
      json_value.total_amount,
      json_value.transaction_timestamp,
      json_value.customer_id,
      explode(array_distinct(json_value.items)) as item
from gizmobox.silver.orders_json

In [0]:
%sql
select * from tv_orders_exploded;

In [0]:
%sql
select 
   order_id,
   order_status,
   payment_method,
   total_amount,
   transaction_timestamp,
   customer_id,
   item.item_id,
   item.name,
   item.price,
   item.quantity,
   item.category,
   item.details.brand,
   item.details.color
from tv_orders_exploded;

In [0]:
%sql
create or replace table gizmobox.silver.orders
as
select 
   order_id,
   order_status,
   payment_method,
   total_amount,
   transaction_timestamp,
   customer_id,
   item.item_id,
   item.name,
   item.price,
   item.quantity,
   item.category,
   item.details.brand,
   item.details.color
from tv_orders_exploded;

In [0]:
%sql
select * from gizmobox.silver.orders;